# Video timing of the Illinois agility test

This notebook is the entry point of the project.
It measures the time of a test (start and finish on the same line)
from a video, by manually selecting the start frame and the end frame.

## Step 1: read the information of a video

In [48]:
# --- Libraries used in the project ---

# OpenCV: library for image and video processing
import cv2

# pathlib: handles file paths so they work
# on Windows, Mac and Linux (cross-platform requirement)
from pathlib import Path

# numpy: arrays (an image is an array of pixels)
import numpy as np

# pandas: tables of data (read/write CSV files)
import pandas as pd

In [49]:
# --- Project folder paths ---

# Folder containing the videos (path RELATIVE to the notebook,
# so it works on any computer)
video_folder = Path("Video")

# Check that the folder exists and list the videos it contains
print("Folder exists:", video_folder.exists())
for file in video_folder.iterdir():
    print(file.name)

Folder exists: True
.DS_Store
ADELE_LIMBOURG.MOV
ARCHIBALD_MONTILLET.MOV
BAPTISTE_MAUFFAIT.MOV


In [50]:
# --- Function: read the fps and number of frames of a video ---

def get_video_info(video_path):
    """
    Open a video and return its fps and number of frames.

    video_path : path to the video file (Path object)
    returns    : (fps, n_frames)
    """
    # Open the video: "video" represents the file
    video = cv2.VideoCapture(str(video_path))

    # If the path is wrong, OpenCV does not raise an error by itself:
    # so we check ourselves that the video is opened
    if not video.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    # Read the frames per second stored in the file
    fps = video.get(cv2.CAP_PROP_FPS)

    # Read the total number of frames (OpenCV returns a float, convert to int)
    n_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    # Close the video to release the file
    video.release()

    # Return both values
    return fps, n_frames

In [51]:
# --- Test the function on one video ---

# Replace "YOUR_VIDEO_NAME.mov" with one of the names printed in cell 3
test_path = video_folder / "ADELE_LIMBOURG.MOV"



# Call the function: it returns 2 values stored in 2 variables
fps, n_frames = get_video_info(test_path)

# Computed duration = number of frames / frames per second
duration = n_frames / fps

print(f"fps (from file)  : {fps}")
print(f"number of frames : {n_frames}")
print(f"computed duration: {duration:.2f} s")

fps (from file)  : 30.0
number of frames : 5522
computed duration: 184.07 s


## Step 2: ask the user for the real fps

Slow-motion videos are recorded at 240 fps but stored at 30 fps in the file.
The fps read from the file is therefore wrong: the user must give the real one.

In [52]:
# --- Function: ask the user for the real recording fps ---

def ask_fps(file_fps, default_fps=240):
    """
    Ask the user for the real recording fps of the video.

    file_fps    : fps read from the file (shown as information only)
    default_fps : value used if the user just presses Enter
    returns     : the real fps (float)
    """
    # Show the fps stored in the file so the user can compare
    print(f"fps stored in the file: {file_fps}")

    # Repeat the question until the answer is valid
    while True:
        # input() shows a message and returns what the user typed, as TEXT
        answer = input(f"Real recording fps (press Enter for {default_fps}): ")

        # If the user typed nothing, keep the default value
        if answer.strip() == "":
            return float(default_fps)

        # Try to convert the text into a number
        try:
            fps = float(answer)
        except ValueError:
            # The text is not a number (e.g. "abc"): ask again
            print("Please type a number (e.g. 240).")
            continue

        # A fps must be positive
        if fps <= 0:
            print("The fps must be greater than 0.")
            continue

        # Valid answer: return it
        return fps

In [53]:
# --- Test: read the video info, then ask for the real fps ---

# fps stored in the file and number of frames (step 1)
file_fps, n_frames = get_video_info(test_path)

# Real fps given by the user (step 2)
real_fps = ask_fps(file_fps)

# Real duration of the video = number of frames / real fps
real_duration = n_frames / real_fps

print(f"real fps           : {real_fps}")
print(f"real video duration: {real_duration:.2f} s")

fps stored in the file: 30.0
real fps           : 240.0
real video duration: 23.01 s


## Step 3: measure the test time in the video

One window shows the video with a progress bar.
- Enter (1st): START frame
- Enter (2nd): END frame
- Enter (3rd): save the result

In [ ]:
# --- Key codes (arrows are different on each operating system) ---
#                 Mac     Windows   Linux
KEY_RIGHT = {63235, 2555904, 65363}
KEY_LEFT  = {63234, 2424832, 65361}
KEY_UP    = {63232, 2490368, 65362}
KEY_DOWN  = {63233, 2621440, 65364}
KEY_ENTER = {13, 10}
KEY_BACK  = {8, 127}      # Backspace (Windows/Linux = 8, Mac = 127)
KEY_ESC   = {27}

BAR_HEIGHT = 40           # height of the progress bar area (pixels)
BAR_MARGIN = 10           # space on the left/right of the bar (pixels)


def get_name_from_file(video_path):
    """
    Return the child's full name from the file name, in one piece.
    Example: "VICTOR_DJALAL_ROOS.MOV" -> "VICTOR_DJALAL_ROOS"
             "ELIES OSTA.MOV"         -> "ELIES_OSTA"
    """
    # .stem = file name without extension; spaces replaced by "_"
    return video_path.stem.strip().replace(" ", "_")


def put_text(img, text, position, color, size=0.8):
    """Write text with a black outline so it is readable on any background."""
    cv2.putText(img, text, position, cv2.FONT_HERSHEY_SIMPLEX, size, (0, 0, 0), 4)
    cv2.putText(img, text, position, cv2.FONT_HERSHEY_SIMPLEX, size, color, 2)


def frame_to_x(index, n_frames, width):
    """Convert a frame number into a horizontal position on the bar."""
    return BAR_MARGIN + int(index / (n_frames - 1) * (width - 2 * BAR_MARGIN))


def draw_display(frame, index, n_frames, fps, start, end):
    """
    Build the image shown in the window: video frame + texts + progress bar.
    Colors are in BGR order (Blue, Green, Red) in OpenCV.
    """
    h, w = frame.shape[:2]

    # Add a black band under the frame for the progress bar
    band = np.zeros((BAR_HEIGHT, w, 3), dtype=np.uint8)
    img = np.vstack([frame, band])

    # Progress bar: grey background, light part = where we are
    y_top, y_bottom = h + 10, h + BAR_HEIGHT - 10
    cv2.rectangle(img, (BAR_MARGIN, y_top), (w - BAR_MARGIN, y_bottom), (80, 80, 80), -1)
    x_now = frame_to_x(index, n_frames, w)
    cv2.rectangle(img, (BAR_MARGIN, y_top), (x_now, y_bottom), (200, 200, 200), -1)

    # Marks on the bar: green line = START, red line = END
    for mark, color in ((start, (0, 255, 0)), (end, (0, 0, 255))):
        if mark is not None:
            x = frame_to_x(mark, n_frames, w)
            cv2.line(img, (x, h + 2), (x, h + BAR_HEIGHT - 2), color, 3)

    # Line 1: frame number and video time
    put_text(img, f"frame {index}/{n_frames - 1} | video {index / fps:.3f} s",
             (20, 35), (255, 255, 255))

    # Line 2: depends on the step (no start yet / running / finished)
    if start is None:
        put_text(img, "Press ENTER on the START frame", (20, 75), (0, 255, 255))
    elif end is None:
        put_text(img, f"START  |  test time: {(index - start) / fps:.3f} s",
                 (20, 75), (0, 255, 0), 1.0)
    else:
        put_text(img, f"END  |  test time: {(end - start) / fps:.3f} s",
                 (20, 75), (0, 0, 255), 1.0)
        put_text(img, "ENTER = save | BACKSPACE = undo | ESC = cancel",
                 (20, 115), (255, 255, 255), 0.6)

    return img

In [ ]:
# --- Function: measure the test time of one video ---

def measure_test(video_path, real_fps, scale=0.5):
    """
    Open the video, let the user mark START and END, return the result.

    returns : dict with name, frames and time (s),
              or None if cancelled with Escape
    """
    video = cv2.VideoCapture(str(video_path))
    if not video.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")
    n_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    # "state" is shared with the mouse function below
    state = {"index": 0, "frame_w": 1, "frame_h": 1}

    def on_mouse(event, x, y, flags, param):
        """Click or drag on the bar to jump in the video."""
        if y < state["frame_h"]:
            return  # click on the image, not on the bar: ignore
        clicked = event == cv2.EVENT_LBUTTONDOWN
        dragged = event == cv2.EVENT_MOUSEMOVE and (flags & cv2.EVENT_FLAG_LBUTTON)
        if clicked or dragged:
            ratio = (x - BAR_MARGIN) / (state["frame_w"] - 2 * BAR_MARGIN)
            ratio = max(0.0, min(ratio, 1.0))
            state["index"] = int(round(ratio * (n_frames - 1)))

    window = "Measure test"
    cv2.namedWindow(window)
    cv2.setMouseCallback(window, on_mouse)

    start, end = None, None
    last_index, frame = None, None
    saved = False

    while True:
        # Keep the index inside the video
        state["index"] = max(0, min(state["index"], n_frames - 1))
        index = state["index"]

        # Read a new frame only if the position changed (faster)
        if index != last_index:
            video.set(cv2.CAP_PROP_POS_FRAMES, index)
            ok, raw = video.read()
            if not ok:
                state["index"] = max(index - 1, 0)
                continue
            frame = cv2.resize(raw, None, fx=scale, fy=scale)
            state["frame_h"], state["frame_w"] = frame.shape[:2]
            last_index = index

        cv2.imshow(window, draw_display(frame, index, n_frames, real_fps, start, end))

        # Wait 30 ms for a key (-1 if none), so the mouse bar stays reactive
        key = cv2.waitKeyEx(30)

        if key in KEY_RIGHT:
            state["index"] += 1
        elif key in KEY_LEFT:
            state["index"] -= 1
        elif key in KEY_UP:
            state["index"] += 10
        elif key in KEY_DOWN:
            state["index"] -= 10
        elif key in KEY_ENTER:
            if start is None:
                start = index                       # 1st Enter: START
            elif end is None:
                if index > start:
                    end = index                     # 2nd Enter: END
                else:
                    print("END must be after START.")
            else:
                saved = True                        # 3rd Enter: save
                break
        elif key in KEY_BACK:
            if end is not None:
                end = None                          # undo END
            elif start is not None:
                start = None                        # undo START
        elif key in KEY_ESC:
            break                                   # cancel
        elif key != -1:
            print("Unknown key code:", key)

    # Close the video and the window (extra waitKey calls needed on Mac)
    video.release()
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)

    if not saved:
        return None

    return {
        "name": get_name_from_file(video_path),
        "start_frame": start,
        "end_frame": end,
        "fps": real_fps,
        "time_s": round((end - start) / real_fps, 3),
    }

In [ ]:
# --- Test: measure one video ---

result = measure_test(test_path, real_fps)

if result is None:
    print("Measurement cancelled.")
else:
    print(f"{result['name']} : {result['time_s']:.3f} s")

Adele Limbourg : 5.021 s


## Step 4: measure all the videos and save the results

The program opens each video one after the other.
Results are saved in `results/video_measures.csv` after EACH video,
so the session can be stopped (Escape) and resumed later:
videos already measured are skipped.

In [ ]:
# --- Paths of the results ---
results_folder = Path("results")
results_folder.mkdir(exist_ok=True)          # create the folder if it does not exist
results_csv = results_folder / "video_measures.csv"

# Columns of the results table
COLUMNS = ["name", "video", "start_frame", "end_frame", "fps", "time_s"]

# Video file extensions accepted (compared in lower case)
VIDEO_EXTENSIONS = {".mov", ".mp4", ".m4v", ".avi"}


def list_videos(folder):
    """Return the list of video files in the folder, sorted by name."""
    videos = [f for f in folder.iterdir() if f.suffix.lower() in VIDEO_EXTENSIONS]
    return sorted(videos)


def load_results(csv_path):
    """Read the results file if it exists, otherwise return an empty table."""
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return pd.DataFrame(columns=COLUMNS)


def save_result(result, csv_path):
    """
    Add one result to the CSV file.
    If the video was already measured, its old line is replaced.
    """
    table = load_results(csv_path)
    table = table[table["video"] != result["video"]]          # remove old line
    new_line = pd.DataFrame([result], columns=COLUMNS)
    table = pd.concat([table, new_line], ignore_index=True)   # add new line
    table = table.sort_values("video")
    table.to_csv(csv_path, index=False)


def measure_all_videos(folder, csv_path, real_fps, redo=False):
    """
    Measure every video of the folder one after the other.

    redo=False : skip videos already in the results file
    redo=True  : measure all videos again
    Escape in the video window stops the session.
    """
    videos = list_videos(folder)
    done = set(load_results(csv_path)["video"])

    for i, video_path in enumerate(videos, start=1):
        if video_path.name in done and not redo:
            print(f"[{i}/{len(videos)}] {video_path.name}: already measured, skipped")
            continue

        print(f"[{i}/{len(videos)}] {video_path.name}: measuring...")
        result = measure_test(video_path, real_fps)

        if result is None:
            print("Session stopped. Results saved so far are kept.")
            break

        result["video"] = video_path.name
        save_result(result, csv_path)
        print(f"   -> {result['name']}: {result['time_s']:.3f} s (saved)")
        
    print("Done.")

In [58]:
# --- Run the measurements on all videos ---

# Ask the real fps once (all videos are filmed with the same setting)
first_video = list_videos(video_folder)[0]
file_fps, _ = get_video_info(first_video)
real_fps = ask_fps(file_fps)

measure_all_videos(video_folder, results_csv, real_fps)

# Show the results table
load_results(results_csv)

fps stored in the file: 30.0
[1/3] ADELE_LIMBOURG.MOV: already measured, skipped
[2/3] ARCHIBALD_MONTILLET.MOV: already measured, skipped
[3/3] BAPTISTE_MAUFFAIT.MOV: already measured, skipped
Done.


,video,first_name,last_name,start_frame,end_frame,fps,time_s
0,ADELE_LIMBOURG.MOV,Adele,Limbourg,822,1545,240.0,3.013
1,ARCHIBALD_MONTILLET.MOV,Archibald,Montillet,790,1563,240.0,3.221
2,BAPTISTE_MAUFFAIT.MOV,Baptiste,Mauffait,2356,3541,240.0,4.938
